In [ ]:
! pip install torch torchvision torchaudio

In [ ]:
# installation des bibliothèques pré-requises
!pip install rasterio shapely geopandas earthpy cartopy

In [ ]:
# pré-requis

import shutil

import os
import glob as glob
import numpy as np
import rasterio as rio
from rasterio import plot
from rasterio.plot import show
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.cm import get_cmap
from matplotlib.axes import Axes
from IPython.display import HTML
from skimage import exposure

from rasterio.mask import mask
from shapely.geometry import mapping
import matplotlib.pyplot as plt
import geopandas as gpd
import earthpy as et
import earthpy.plot as ep
import earthpy.spatial as es
import cartopy as cp

import pandas as pd

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform

In [ ]:
REPERTOIRE_DATASET = "D:\\DATA\\FOLDER"

REPERTOIRE_SATELLITE = "S-2.OK"
#REPERTOIRE_SATELLITE = "Last_S-2.KO-OK" # les éléments ayant raté
#REPERTOIRE_SATELLITE = "Landsat_9"


EXTENSION_SENTINEL2 = ".jp2"
EXTENSION_LANDSAT9 = ".TIF"
EXTENSION_PLANET = ".tif"
EXTENSION_FUSION = ".tif"
EXTENSION_IMAGES_SAT = EXTENSION_SENTINEL2

REPERTOIRE_COMPLET_SATELLITE =  REPERTOIRE_DATASET + "\\" + REPERTOIRE_SATELLITE
REPERTOIRE_SATELLITE_DECOUPE = "S-2_DECOUPE_FINAL" # AJOUTER 3ha.old
#REPERTOIRE_SATELLITE_DECOUPE = "S-2_DECOUPE_KO-OK"
#REPERTOIRE_SATELLITE_DECOUPE = "L-9_DECOUPE_FINAL" # AJOUTER 3ha.old

REPERTOIRE_COMPLET_SATELLITE_DECOUPE = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SATELLITE_DECOUPE
print("Nom satellite")
print(REPERTOIRE_SATELLITE)
print("Extension image satellite")
print(EXTENSION_IMAGES_SAT)
print("Répertoire complet satellite")
print(REPERTOIRE_COMPLET_SATELLITE)
print("Répertoire complet satellite découpé")
print(REPERTOIRE_COMPLET_SATELLITE_DECOUPE)

REPERTOIRE_SHAPE_3ha = "ROI_3ha"
REPERTOIRE_COMPLET_SHAPE_3ha = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SHAPE_3ha
print("Répertoire complet shape 3ha")
print(REPERTOIRE_COMPLET_SHAPE_3ha)


In [ ]:
# création du dossier de destination
if not os.path.exists(REPERTOIRE_COMPLET_SATELLITE_DECOUPE):
    os.mkdir(REPERTOIRE_COMPLET_SATELLITE_DECOUPE)

listes_id = os.listdir(REPERTOIRE_COMPLET_SATELLITE)
num_obs = 0
for id in listes_id:
    print("Traitement de l'observation " + str(id))
    # récupération de la liste des images de l'obs id
    dossier_id_source = REPERTOIRE_COMPLET_SATELLITE + "\\" + str(id)
    liste_images_sat = os.listdir(dossier_id_source)

    # création du dossier id dans le repertoire des images découpées
    dossier_id_dest = REPERTOIRE_COMPLET_SATELLITE_DECOUPE + "\\" + str(id)
    if not os.path.exists(dossier_id_dest):
        os.mkdir(dossier_id_dest)

    # découpage des images
    for image in liste_images_sat:
        split = os.path.splitext(image)
        extension = split[1]
        if extension == EXTENSION_IMAGES_SAT: # pour ne pas considérer les fichiers non image
            #print(image)
            
            #lecture du fichier image
            chemin_image = REPERTOIRE_COMPLET_SATELLITE + "\\" + str(id)  + "\\" + image
            src = rio.open(chemin_image)
            
            #lecture des shp
            shp = gpd.read_file(REPERTOIRE_COMPLET_SHAPE_3ha + "\\" + str(id) + '.shp')

            # projection du shapefile en concordance avec l'image satellite
            profil_crs_img = src.profile
            shp_utm31N = shp.to_crs(profil_crs_img["crs"])
            
            #découpage suivant le shp
            img_decoupee, metadonnees = es.crop_image(src, shp_utm31N)
            
            # mise à jour des métadonnées
            # img_decoupee.shape #  (bands, rows, columns)
            metadonnees.update({"driver": "GTiff",
                             "height": img_decoupee.shape[1],
                             "width": img_decoupee.shape[2],
                             "transform": metadonnees["transform"]})
        
            # Save to disk so you can use the file later.
            chemin_image_decoupee = dossier_id_dest + "\\" + image
            with rio.open(chemin_image_decoupee, 'w', **metadonnees) as ff:
                ff.write(img_decoupee[0], 1)
    num_obs += 1

print("Nombre d'observations traitées au total", num_obs)

In [ ]:
print(id) # si erreur
